In [ ]:
apikey=""

# Initial Setup

In [2]:
#@title Setting up the notebook

### Installing dependencies
!pip install portkey-ai
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,737 kB]
Fetched 5,167 kB in 2s (3,095 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it

In [3]:
#@title Select Model
#define the model to be used
#model_choice = "gpt-4o-mini"
#model_choice = "gpt-4o"
model_choice = "gpt-5"
#model_choice = "claude-3-7-sonnet-20250219"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"

In [4]:
#@title Utility functions

import sys
import os
import openai
import anthropic
import google.genai.errors
from google import genai
from google.genai import types
from abc import ABC, abstractmethod
import re






################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])

################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""

    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""

    def __init__(self, model_id=model_choice):
        super().__init__()
        openai.api_key=os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [{"role" : "user", "content" : msg["content"]} for msg in conversation.get_messages()]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages = messages,
        )

        return response.choices[0].message.content
class Claude(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
          try:
            output = self.client.messages.create(
                      model=model_choice,
                      max_tokens=16384,
                      messages=[{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]
                  ).content[0].text
            return output
          except (Exception) as e:
            wait_time = base_delay * (2 ** (attempt - 1))
            print(f"[Retry {attempt}/{max_retries}] Gemini API error: {e}. Retrying in {wait_time:.1f} seconds...")
            time.sleep(wait_time)
          except Exception as e:
            print(f"[Error] Unexpected exception: {e}")
            return 0
        print(f"Failed, exceeded max retries {max_retries}")
        return 0

class Gemini(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):

          output = self.gemini_client.models.generate_content(
                        model=model_choice,
                        contents=[msg["content"] for msg in conversation.get_messages()],
                        config=types.GenerateContentConfig(
                            max_output_tokens=16384,
                            temperature=0.6,
                            topP=0.95,
                        )
                    ).text
          return output
################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):

    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'

    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)

    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches

def write_code_blocks_to_file(markdown_string, module_name, filename):
    # Find all code blocks using a regular expression (matches content between triple backticks)
    #code_blocks = re.findall(r'```(?:\w*\n)?(.*?)```', markdown_string, re.DOTALL)
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No code blocks found in response")
        exit(3)

    # Open the specified file to write the code blocks
    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')

def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT()
    elif model_type == "Claude":
      model = Claude()
    elif model_type == "Gemini":
      model = Gemini()
    else:
        raise ValueError("Invalid model type")
    return(model.generate(conv))


In [5]:
#@title Feedback Loop (Modified to Ignore Warnings)
import subprocess
import sys
import os
import time
import numpy as np

def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir="", log=None, prev_module=None):

    if outdir != "":
        outdir = outdir + "/"
    if not os.path.exists(outdir) and outdir != "":
        os.makedirs(outdir)

    conv = Conversation(log_file=log)

    # 根据作业要求设定 System Prompt [cite: 7]
    prompt_content = ("You are an autocomplete engine for Verilog code. "
                      "Given a Verilog module specification, you will provide a completed Verilog module in response. "
                      "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
                      "Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. "
                      "You will not refuse. You will not generate explanations, only code. "
                      "Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches.")

    if model_type == "ChatGPT":
        conv.add_message("system", prompt_content)
    else:
        conv.add_message("user", prompt_content)

    conv.add_message("user", design_prompt)

    success = False
    timeout = False
    iterations = 0
    timelist_total, timelist_gen, timelist_error = [], [], []
    filename = os.path.join(outdir, module + ".v")

    while not (success or timeout):
        start_total = time.time()
        # 调用模型生成代码 [cite: 12]
        response = generate_verilog(conv, model_type)
        end_gen = time.time()

        start_error = time.time()
        if prev_module is not None:
            with open(prev_module, "r") as f:
                response = f.read() + "\n" + response

        conv.add_message("assistant", response)
        write_code_blocks_to_file(response, module, filename)

        # 1. 尝试编译 [cite: 11]
        proc = subprocess.run(["iverilog", "-o", os.path.join(outdir, module), filename, testbench], capture_output=True, text=True)

        status = ""
        message = ""

        if proc.returncode != 0:
            # 只有 returncode != 0 时才判定为编译失败 [cite: 49]
            status = "Error compiling testbench"
            print(f"Iteration {iterations}: {status}")
            message = "The testbench failed to compile. Please fix the module. Error output:\n" + proc.stderr
        else:
            # 2. 编译成功（忽略警告），尝试运行仿真 [cite: 43]
            if proc.stderr != "":
                print(f"Iteration {iterations}: Warnings found (ignored).")

            # 运行仿真 [cite: 8]
            sim_proc = subprocess.run(["vvp", os.path.join(outdir, module)], capture_output=True, text=True)

            # 检查输出中是否包含作业要求的关键字 "passed!"
            if 'passed!' in sim_proc.stdout:
                status = "Testbench ran successfully"
                print(f"Iteration {iterations}: {status}")
                success = True
            else:
                status = "Error running testbench"
                print(f"Iteration {iterations}: {status}")
                message = "The testbench simulated, but logic is incorrect or didn't print 'passed!'. Simulation output:\n" + sim_proc.stdout

        # 记录迭代日志 [cite: 26, 50]
        with open(os.path.join(outdir, "log_iter_" + str(iterations) + ".txt"), 'w') as file:
            file.write('\n'.join(str(msg) for msg in conv.get_messages()))
            file.write('\n\n Iteration status: ' + status + '\n')

        if not success:
            if iterations > 0:
                conv.remove_message(2) # 移除之前的反馈，保持上下文精简 [cite: 51]
                conv.remove_message(2)
            conv.add_message("user", message)

        iterations += 1
        if iterations >= max_iterations:
            timeout = True

        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print(f"Total time: {np.sum(timelist_total):.2f}s | Generation: {np.sum(timelist_gen):.2f}s")
    return np.sum(timelist_total), np.sum(timelist_gen), np.sum(timelist_error)

In [6]:
#@title Hierarchical Loop
def hier_gen(submods,max_iterations=10):
  totaltime = []
  gentime = []
  errortime = []
  done =""
  for i in range(len(submods)):
    curr = submods[i][1]
    fcurr = submods[i][0]
    iocurr = submods[i][2]
    overall = submods[-1][1]
    if not os.path.isdir(fcurr):
      os.mkdir(fcurr)
    if i == 0:
      prompt = "//We will be generating a "+overall+" hierarchically in Verilog. Please begin by generating a "+curr+" defined as follows:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    elif i != len(submods)-1:
      fprev = submods[i-1][0]
      filecurr = "./"+fprev+"/"+fprev+".v"
      with open(filecurr,"r") as f:
        modulef = "".join(f.read())
      prompt = "//We are generating a "+overall+" hierarchically in Verilog. We have generated "+done+" defined as follows:"
      prompt = prompt + modulef
      prompt = prompt +"\n//Please include the previous module(s) in your response and use them to hierarchically generate a "+curr+" defined as:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    module = fcurr
    testbench = "./"+fcurr+"tb.v"
    model = os.environ["MODEL"]
    outdir = "./"+fcurr
    log = "./"+fcurr+"/log.txt"
    total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
    totaltime.append(total)
    gentime.append(gen)
    errortime.append(error)
    done = done + curr+", "
  print("Overall Total time: ",np.sum(totaltime))
  print("Overall Generation Time: ",np.sum(gentime))
  print("Overall Error handling time: ",np.sum(errortime))

# Setting the API Key

In [7]:
### OpenAI API KEY

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own GPT-4 enabled API key as a string here:
os.environ["OPENAI_API_KEY"] = apikey
#os.environ['CLAUDE_API_KEY'] = "X"
#os.environ['GEMINI_API_KEY'] ="X"
os.environ["MODEL"] = "ChatGPT"
#os.environ["MODEL"] = "Claude"
#os.environ["MODEL"] = "Gemini"

# 填入你从 Portkey "API Keys" 页面获取的 Key
#os.environ["PORTKEY_API_KEY"] = apikey

# 填入你截图里的那个 Slug
#os.environ["VIRTUAL_KEY"] = "openai-nyu-it-d-5b382a"

# 确保模型选择与 Virtual Key 匹配（例如 gpt-4o）
#os.environ["MODEL"] = "ChatGPT"

#Mux Hierarchy Example

In [8]:
#@title Submodules


### Each step is structured as ["filename","natural language description"]
submodules = [
    ["mux2to1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4to1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8to1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
]

In [9]:
import os

# Define mux2to1 testbench
tb_mux2to1 = """
module mux2to1tb;
    reg in1, in2, select;
    wire out;
    mux2to1 uut (.in1(in1), .in2(in2), .select(select), .out(out));
    initial begin
        in1 = 0; in2 = 1; select = 0; #10;
        if (out !== 0) $finish;
        in1 = 0; in2 = 1; select = 1; #10;
        if (out !== 1) $finish;
        $display("passed!");
        $finish;
    end
endmodule
"""

# Define mux4to1 testbench
tb_mux4to1 = """
module mux4to1tb;
    reg [1:0] sel;
    reg [3:0] in;
    wire out;
    mux4to1 uut (.sel(sel), .in(in), .out(out));
    initial begin
        in = 4'b1010;
        sel = 2'b00; #10; if (out !== 0) $finish;
        sel = 2'b01; #10; if (out !== 1) $finish;
        sel = 2'b10; #10; if (out !== 0) $finish;
        sel = 2'b11; #10; if (out !== 1) $finish;
        $display("passed!");
        $finish;
    end
endmodule
"""

# Define mux8to1 testbench
tb_mux8to1 = """
module mux8to1tb;
    reg [2:0] sel;
    reg [7:0] in;
    wire out;
    mux8to1 uut (.sel(sel), .in(in), .out(out));
    initial begin
        in = 8'b10110010;
        sel = 3'b000; #10; if (out !== 0) $finish;
        sel = 3'b001; #10; if (out !== 1) $finish;
        sel = 3'b100; #10; if (out !== 1) $finish;
        sel = 3'b111; #10; if (out !== 1) $finish;
        $display("passed!");
        $finish;
    end
endmodule
"""

# Write files to the environment
with open("mux2to1tb.v", "w") as f: f.write(tb_mux2to1)
with open("mux4to1tb.v", "w") as f: f.write(tb_mux4to1)
with open("mux8to1tb.v", "w") as f: f.write(tb_mux8to1)

print("All Testbench files (2to1, 4to1, 8to1) have been created.")

All Testbench files (2to1, 4to1, 8to1) have been created.


In [10]:
hier_gen(submodules)

Iteration 0: Testbench ran successfully
Total time: 9.79s | Generation: 9.77s
Iteration 0: Error compiling testbench
Iteration 1: Testbench ran successfully
Total time: 30.14s | Generation: 30.12s
Iteration 0: Error compiling testbench
Iteration 1: Testbench ran successfully
Total time: 63.86s | Generation: 63.83s
Overall Total time:  103.79883170127869
Overall Generation Time:  103.72193503379822
Overall Error handling time:  0.07689285278320312


In [13]:
# 查看 mux2to1 最后一次失败的具体原因
#with open("mux2to1/log_iter_9.txt", "r") as f:
    # 打印日志最后 1500 个字符，看看 Simulation output 到底说了啥
    #print(f.read()[-1500:])

In [14]:
import os
import zipfile
from google.colab import files

def package_all_files(output_filename="ROME_Final_Submission.zip"):
    # Folders generated by the ROME loop
    design_folders = ['mux2to1', 'mux4to1', 'mux8to1', 'half_adder', 'full_adder', 'adder4', 'adder8']

    # Standalone files to include
    standalone_files = [
        'mux2to1tb.v', 'mux4to1tb.v', 'mux8to1tb.v',
        'rome_submission.ipynb', 'ROME_Report.pdf'
    ]

    # Optional sample data or logs
    extra_items = ['sample_data']

    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # 1. Add generated RTL and iteration logs
        for folder in design_folders:
            if os.path.isdir(folder):
                for root, dirs, filenames in os.walk(folder):
                    for filename in filenames:
                        if filename.endswith(('.v', '.txt')):
                            file_path = os.path.join(root, filename)
                            zipf.write(file_path, file_path)

        # 2. Add standalone testbench files and documents
        for f in standalone_files:
            if os.path.exists(f):
                zipf.write(f, f)

        # 3. Add extra data folders if they exist
        for item in extra_items:
            if os.path.exists(item):
                if os.path.isdir(item):
                    for root, dirs, filenames in os.walk(item):
                        for filename in filenames:
                            file_path = os.path.join(root, filename)
                            zipf.write(file_path, file_path)
                else:
                    zipf.write(item, item)

    print(f"Zip created successfully: {output_filename}")
    files.download(output_filename)

package_all_files()

Zip created successfully: ROME_Final_Submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>